# Bitonic Sort on PYNQ-Z2

Benchmarks the `sort_top` HLS IP against NumPy for **N = 64** 32-bit integers.

**Data path:** PS → DMA MM2S → AXI-Stream → `sort_top` → AXI-Stream → DMA S2MM → PS

| Register (s_axi_control) | Offset |
|--------------------------|--------|
| `ap_ctrl` (bit0=start, bit1=done, bit2=idle) | `0x00` |

## 1. Load Overlay

In [ ]:
import time
import numpy as np
from pynq import Overlay, allocate

N = 64

ol = Overlay('bitonic_design.bit')
print('Overlay loaded')
print('IP blocks:', list(ol.ip_dict.keys()))

dma      = ol.axi_dma_0
sort_ip  = ol.sort_top_0
hw_timer = ol.axi_timer_0

## 2. Helper Functions

In [ ]:
AP_CTRL        = 0x00
FLAG_POST_DMA  = 0x10  # verify against HLS synthesis report (xsort_top_hw.h)
FLAG_POST_SORT = 0x18  # verify against HLS synthesis report
TCSR0, TLR0, TCR0 = 0x00, 0x04, 0x08
FCLK_MHZ = 50.0  # PS drives FCLK0 at 50 MHz (20 ns period)


def timer_start(tmr):
    tmr.write(TLR0, 0)
    tmr.write(TCSR0, 0x020)  # load
    tmr.write(TCSR0, 0x080)  # enable, count up


def timer_read(tmr):
    return tmr.read(TCR0)


def timer_stop(tmr):
    cycles = tmr.read(TCR0)
    tmr.write(TCSR0, 0x000)
    return cycles


def bitonic_sort_hw(data, timed=False):
    """Sort data using the FPGA bitonic sort IP via AXI DMA."""
    assert len(data) == N, f'IP is hardcoded for N={N}'

    in_buf  = allocate(shape=(N,), dtype=np.int32)
    out_buf = allocate(shape=(N,), dtype=np.int32)
    np.copyto(in_buf, data.astype(np.int32))
    out_buf[:] = 0

    if timed:
        timer_start(hw_timer)

    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    sort_ip.write(AP_CTRL, 0x01)  # ap_start

    if timed:
        # poll until DMA-in completes (flag_post_dma = 1) → sort is starting
        while not sort_ip.read(FLAG_POST_DMA):
            pass
        t_dma_done = timer_read(hw_timer)

        # poll until sort completes (flag_post_sort = 1)
        while not sort_ip.read(FLAG_POST_SORT):
            pass
        t_sort_done = timer_read(hw_timer)

        sort_cycles = t_sort_done - t_dma_done

    dma.sendchannel.wait()
    dma.recvchannel.wait()

    result = np.array(out_buf, dtype=np.int32)
    in_buf.freebuffer()
    out_buf.freebuffer()

    return (result, sort_cycles) if timed else result

## 3. Correctness Check

In [ ]:
rng = np.random.default_rng(42)
all_pass = True

# data_t = ap_uint<16> → values must be in [0, 65535]
print('Random trials:')
for trial in range(5):
    data   = rng.integers(0, 65536, size=N, dtype=np.int32)
    hw_out = bitonic_sort_hw(data)
    ok     = np.array_equal(hw_out & 0xFFFF, np.sort(data & 0xFFFF))
    all_pass &= ok
    print(f'  trial {trial+1}: {"PASS" if ok else "FAIL"}')

print('Edge cases:')
edge_cases = [
    ('ascending',  np.arange(N,            dtype=np.int32)),
    ('descending', np.arange(N-1, -1, -1,  dtype=np.int32)),
    ('all-same',   np.full(N, 7,            dtype=np.int32)),
    ('all-max',    np.full(N, 65535,         dtype=np.int32)),
    ('mixed',      rng.integers(0, 65536, size=N, dtype=np.int32)),
]
for label, data in edge_cases:
    ok = np.array_equal(bitonic_sort_hw(data) & 0xFFFF, np.sort(data & 0xFFFF))
    all_pass &= ok
    print(f'  {label:<12}: {"PASS" if ok else "FAIL"}')

print()
print('Overall:', 'PASS ✓' if all_pass else 'FAIL ✗')

## 4. Timing Benchmark

Sort time is isolated using two AXI-Lite flags written by the IP mid-execution:

| Flag | Offset | When set |
|------|--------|----------|
| `flag_post_dma` | `0x10` | 1 cycle after last AXI-Stream word consumed (sort start) |
| `flag_post_sort` | `0x18` | 1 cycle after `bitonic_sort()` returns (sort end) |

The AXI timer runs free-running from before `ap_start`. The PS polls each flag and reads the timer on transition. **Sort cycles = t_sort_done − t_dma_done**, excluding all DMA overhead.

> Note: verify `FLAG_POST_DMA` / `FLAG_POST_SORT` offsets against `xsort_top_hw.h` in the HLS driver output.

In [ ]:
RUNS = 20
sort_cycles_list = []
wall_us_list     = []
sw_us_list       = []

for _ in range(RUNS):
    data = rng.integers(0, 65536, size=N, dtype=np.int32)

    # HW run — sort-only timing via AXI-Lite flags
    t0 = time.perf_counter()
    _, sort_cycles = bitonic_sort_hw(data, timed=True)
    wall_us_list.append((time.perf_counter() - t0) * 1e6)
    sort_cycles_list.append(sort_cycles)

    # SW baseline (numpy)
    t0 = time.perf_counter()
    np.sort(data)
    sw_us_list.append((time.perf_counter() - t0) * 1e6)

sort_cycles_arr = np.array(sort_cycles_list)
sort_us         = sort_cycles_arr / FCLK_MHZ
wall_us         = np.array(wall_us_list)
sw_us           = np.array(sw_us_list)

print(f'N = {N}, FCLK = {FCLK_MHZ:.0f} MHz, runs = {RUNS}')
print()
print(f'{"Metric":<32} {"Mean":>10} {"Min":>10} {"Max":>10}')
print('-' * 66)
print(f'{"Sort-only cycles (flags)":<32} {sort_cycles_arr.mean():>10.0f} {sort_cycles_arr.min():>10.0f} {sort_cycles_arr.max():>10.0f}')
print(f'{"Sort-only time µs  (flags)":<32} {sort_us.mean():>10.3f} {sort_us.min():>10.3f} {sort_us.max():>10.3f}')
print(f'{"Wall time µs (total)":<32} {wall_us.mean():>10.2f} {wall_us.min():>10.2f} {wall_us.max():>10.2f}')
print(f'{"np.sort µs":<32} {sw_us.mean():>10.2f} {sw_us.min():>10.2f} {sw_us.max():>10.2f}')
print()
print(f'Sort-only speedup vs np.sort: {sw_us.mean()/sort_us.mean():.1f}x')

## 5. Latency Breakdown

Theoretical minimum latency at 50 MHz (20 ns/cycle):

| Stage | Cycles | Time (µs) |
|-------|--------|-----------|
| DMA in (N=64 words) | 64 | 1.28 |
| Bitonic sort (21 sub-stages, II=1) | 21 | 0.42 |
| DMA out (N=64 words) | 64 | 1.28 |
| **Total** | **149** | **2.98** |

Anything above 2.98 µs is DMA setup and AXI bus overhead.

In [ ]:
SORT_CYCLES_THEORETICAL = 21   # 21 passes, each 1 cycle (fully unrolled)
DMA_CYCLES              = N    # INPUT_LOOP: II=1 pipeline, N iterations

theoretical_sort_us = SORT_CYCLES_THEORETICAL / FCLK_MHZ
measured_sort_us    = sort_us.mean()
poll_overhead_us    = measured_sort_us - theoretical_sort_us

print(f'Theoretical sort  (21 passes × 1 cycle) : {SORT_CYCLES_THEORETICAL:3d} cycles = {theoretical_sort_us:.3f} µs')
print(f'Measured sort     (flag_post_dma→sort)  : {sort_cycles_arr.mean():.0f} cycles = {measured_sort_us:.3f} µs')
print(f'PS polling overhead (approx)            :              {poll_overhead_us:.3f} µs')
print()
print(f'DMA-in  (excluded) : {DMA_CYCLES} cycles = {DMA_CYCLES/FCLK_MHZ:.2f} µs')
print(f'DMA-out (excluded) : {N} cycles = {N/FCLK_MHZ:.2f} µs')

## 6. Distribution Plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels = ['np.sort', 'Sort-only (flags)', 'Wall time (total)']
means  = [sw_us.mean(), sort_us.mean(), wall_us.mean()]
stds   = [sw_us.std(),  sort_us.std(),  wall_us.std()]
colors = ['steelblue', 'seagreen', 'coral']

bars = axes[0].bar(labels, means, yerr=stds, capsize=5,
                   color=colors, alpha=0.8, edgecolor='black')
axes[0].set_ylabel('Time (µs)')
axes[0].set_title(f'Sort latency — N={N} @ {FCLK_MHZ:.0f} MHz')
axes[0].grid(axis='y', alpha=0.3)
for bar, mean in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(means)*0.01,
                 f'{mean:.2f}', ha='center', va='bottom', fontsize=9)

axes[1].hist(sort_cycles_arr, bins=15, color='seagreen', alpha=0.8, edgecolor='black')
axes[1].axvline(SORT_CYCLES_THEORETICAL, color='red', linestyle='--',
                label=f'Theoretical min ({SORT_CYCLES_THEORETICAL} cycles)')
axes[1].set_xlabel('PL clock cycles (sort only)')
axes[1].set_ylabel('Count')
axes[1].set_title('Sort-only cycle distribution')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()